In [38]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [39]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [40]:
import json


def generate_dataset():
    prompt = """
Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task"
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text.strip())

In [41]:
dataset = generate_dataset()
dataset

[{'task': "Write a Python function that parses an AWS S3 bucket name from an S3 URI in the format 's3://bucket-name/key/path' and returns just the bucket name."},
 {'task': "Create a JSON object that represents an AWS IAM policy allowing read-only access to a specific S3 bucket named 'my-data-bucket'."},
 {'task': "Write a regular expression that matches and validates AWS ARN (Amazon Resource Name) format, which follows the pattern 'arn:partition:service:region:account-id:resource'."}]

In [42]:
dataset2 = generate_dataset()
with open("dataset.json", "w") as f:
    json.dump(dataset, f ,indent=2)


In [44]:
def run_prompt(test_case):
    """merger the prompt and the test case input , then return the result """
    prompt = f"""
    please solve the following task:
    {test_case["task"]}
    """
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output



In [45]:
def run_test_case(test_case):
    """calls run_prompt , then grade the result  """
    output = run_prompt(test_case)

    # todo - grade the result 
    score = 10 
    return {
        "output": output,
        "score": score ,
        "test_case": test_case
    }


In [46]:
def run_eval(dataset):
    """merger the prompt and the test case input , then return the result """
    results = []
    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    return results

In [47]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [48]:
print(json.dumps(results, indent=2))


[
  {
    "output": "# AWS S3 Bucket Name Parser\n\nHere's a Python function that parses an S3 bucket name from an S3 URI:\n\n```python\ndef parse_s3_bucket_name(s3_uri: str) -> str:\n    \"\"\"\n    Parse an AWS S3 bucket name from an S3 URI.\n    \n    Args:\n        s3_uri (str): S3 URI in the format 's3://bucket-name/key/path'\n    \n    Returns:\n        str: The bucket name\n    \n    Raises:\n        ValueError: If the URI format is invalid\n    \n    Examples:\n        >>> parse_s3_bucket_name('s3://my-bucket/key/path')\n        'my-bucket'\n        >>> parse_s3_bucket_name('s3://my-bucket/')\n        'my-bucket'\n        >>> parse_s3_bucket_name('s3://my-bucket')\n        'my-bucket'\n    \"\"\"\n    if not s3_uri:\n        raise ValueError(\"S3 URI cannot be empty\")\n    \n    if not s3_uri.startswith('s3://'):\n        raise ValueError(\"S3 URI must start with 's3://'\")\n    \n    # Remove the 's3://' prefix\n    uri_without_prefix = s3_uri[5:]\n    \n    # Split by '/' an